In [1]:
from dataclasses import dataclass
from typing_extensions import Self
from typing import Callable
import numpy as np
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from skimage.metrics import structural_similarity as ssim
from torchvision import transforms
from torch.utils.data import DataLoader
from torch import nn
import matplotlib.pyplot as plt
import torch.optim as optim
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
import torch

# %matplotlib widget

from typing import Optional
import matplotlib.pyplot as plt
import torch

from pileup_ml.detectors.pixels import PixelDetector, PixelModule
from pileup_ml.pixels.hits import PixelDigiEvent
from pileup_ml.pixels.patches import (
    PixelModulePatches, PixelEventPatches, _PixelEventGrid,
    event_hits_to_patches, module_patches_to_hits, event_patches_to_hits)

from pileup_ml.compression.base import CompressedPixelEvent

# TODO:
# Use two matrices for VQ-VAE model lossy data compression
# Divide data into 4 x 4 batches
# Find embeddings using ADCS 4 x 4 batches
# scale patches into smaller scales


In [2]:
import optuna

In [3]:
EVENT_COUNT = 12
BATCH_SIZE = 128
PATCH_SIZE = 4
TRIALS = 5

In [4]:
# Utilities
def _get_vq_loss(recon_x, x, vq_loss):
    recon_loss = F.mse_loss(recon_x, x)
    return recon_loss + vq_loss

In [5]:
# Moduliai ir ju savybes aprasomos pixel_modules json'uose
pixel_modules = PixelModule.read_json('/data/cern/pileup_ml/detid_info/detids_bpix.json', '/data/cern/pileup_ml/detid_info/detids_fpix.json')
pix_det = PixelDetector(pixel_modules)

events_train_all = PixelDigiEvent.read_root('/data/cern/pileup_ml/premixlib2024/0001.root', pix_det)
events_train = events_train_all[:EVENT_COUNT]

events_test_all = PixelDigiEvent.read_root('/data/cern/pileup_ml/premixlib2024/0002.root', pix_det)
events_test = events_test_all[:EVENT_COUNT]

 38%|███▊      | 376/1000 [00:01<00:02, 229.05it/s]


KeyboardInterrupt: 

In [ ]:
# Selecting GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Applying seed for reproducability
seed_value = 123

if device.type == "cuda":
    torch.cuda.manual_seed_all(seed_value)
else:
    torch.cuda.manual_seed(seed_value)

device

device(type='cuda')

In [ ]:
# Patches prepearation
X_patches_adcs = []
X_patches = []

for event in events_train:
    event_hits_adcs = event_hits_to_patches(
        event, 
        patch_size=PATCH_SIZE
    ).as_array()
    X_patches_adcs.append(event_hits_adcs)

# Forming without numpy arrays
for event in events_train:
    event_hits = event_hits_to_patches(
        event, 
        patch_size=PATCH_SIZE
    )
    X_patches.append(event_hits)

X_patches = iter(X_patches)

In [ ]:
def collate_fn(batch: torch.tensor):
    tensors = [torch.tensor(x) for x in batch]    
    tensors_padded = pad_sequence(tensors, batch_first=True)
    tensors_padded = tensors_padded.unsqueeze(1)    
    return tensors_padded

In [ ]:
# X_patches_batches = torch.tensor(X_patches_batches).unsqueeze(1).float()
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Pad(padding=13420),
    transforms.Resize((16485, 16)),
])
train_loader = DataLoader(
    dataset=X_patches_adcs, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate_fn
)

In [ ]:
class VQEmbedding(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int, beta: float):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings

        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.embedding.weight.data.uniform_(-1/self.num_embeddings, 1/self.num_embeddings)
        self.beta = beta

    def forward(self, z):
        # b = batch size
        # c = number of channels
        # h = height
        # w = width
        b, c, h, w = z.shape
        z_channel_last = z.permute(0, 2, 3, 1)
        z_flattened = z_channel_last.reshape(b*h*w, self.embedding_dim)

        # Calculate distances between z and the codebook embeddings |a-b|²
        # Euclidean distance. Form applied for efficiency in deep learning. In literature l2 distance is used.
        distances = (
            torch.sum(z_flattened ** 2, dim=-1, keepdim=True)                 # a²
            + torch.sum(self.embedding.weight.t() ** 2, dim=0, keepdim=True)  # b²
            - 2 * torch.matmul(z_flattened, self.embedding.weight.t())        # -2ab
        )

        # Get the index with the smallest distance
        # Finds minimum indexes at each row. Uses nearest neigboor look up
        encoding_indices = torch.argmin(distances, dim=-1)

        # Get the quantized vector
        z_q = self.embedding(encoding_indices)
        # when we find the indexes feature vectors, we reshape them to the encoder latent 
        # representation shape
        z_q = z_q.reshape(b, h, w, self.embedding_dim)
        
        # changed the order of the reshaped z_q
        z_q = z_q.permute(0, 3, 1, 2)

        # Calculate the commitment loss
        # Measures how closely the encoder outputs and the selected codebook embeddings match each other.
        # loss = torch.mean((z_q.detach()-z)**2) + beta * \
        # torch.mean((z_q - z.detach()) ** 2)
        reconstruction_loss = F.mse_loss(z_q, z.detach())
        commitment_loss = F.mse_loss(z_q.detach(), z)
        loss = reconstruction_loss + self.beta * commitment_loss

        # Straight-through estimator trick for gradient backpropagation
        # Finding closer membership to indexes
        z_q = z + (z_q - z).detach()

        return z_q, loss, encoding_indices

In [ ]:
class Residual(nn.Module):
    """
    Residual Convolutional Layer
    """

    def __init__(self, in_channels, num_hiddens, num_residual_hiddens):
        """
        initialize residual CNN layer

        :param in_channels number: Number of input channels
        :param num_hiddens number: Number of hidden channels
        :param num_residual_hiddens number: Number of residual hiddens
        """
        super(Residual, self).__init__()
        self._block = nn.Sequential(
            nn.ReLU(True),
            # C: 3 -> residual hidden
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=num_residual_hiddens,
                kernel_size=2,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.ReLU(True),
            # C: resiual hidden -> out_hidden
            nn.Conv2d(
                in_channels=num_residual_hiddens,
                out_channels=num_hiddens,
                kernel_size=2,
                stride=1,
                bias=False,
            ),
        )

    def forward(self, x):
        """
        Residual layer

        :param x numpy.ndarray: Input image
        """
        return x + self._block(x)  # residual output


class ResidualStack(nn.Module):
    """
    Residual Convolution Stack
    """

    def __init__(
        self, in_channels, num_hiddens, num_residual_layers, num_residual_hiddens
    ):
        """
        initialize residual stack

        :param in_channels number: Number of input channels
        :param num_hiddens number: Number of hidden channels
        :param num_residual_layers number: Number of residual layers in stack
        :param num_residual_hiddens number: Number of hidden residual channels
        """
        super(ResidualStack, self).__init__()
        self._num_residual_layers = num_residual_layers
        self._layers = nn.ModuleList(
            [
                Residual(in_channels, num_hiddens, num_residual_hiddens)
                for _ in range(self._num_residual_layers)
            ]
        )

    def forward(self, x):
        """
        Apply residual stack

        :param x numpy.ndarray: Input image
        """
        # Apply all residual layers in stack
        for i in range(self._num_residual_layers):
            x = self._layers[i](x)
        return F.relu(x)

In [ ]:
class VQVAE(nn.Module):
    def __init__(
        self, 
        channels: int, 
        latent_dim: int,
        num_embeddings: int,
        num_residual_layers: int,
        num_residual_hiddens: int,
        beta: float
    ):
        super(VQVAE, self).__init__()
        
        self.num_residual_layers = num_residual_layers
        self.num_residual_hiddens = num_residual_hiddens
        self.num_embeddings = num_embeddings

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(channels, 8, kernel_size=2, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(8, 16, kernel_size=2, stride=1, padding=1),
            ResidualStack(
                in_channels=16,
                num_hiddens=16,
                num_residual_layers=num_residual_layers,
                num_residual_hiddens=num_residual_hiddens,
            ),
            nn.ReLU(),
            nn.Conv2d(16, latent_dim, kernel_size=2, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(latent_dim, latent_dim, kernel_size=1)
        )

        # Vector Quantization
        self.vq_layer = VQEmbedding(num_embeddings, latent_dim, beta)
        
        self.vq_residual = ResidualStack(
            in_channels=latent_dim,
            num_hiddens=latent_dim,
            num_residual_layers=latent_dim,
            num_residual_hiddens=num_residual_hiddens,
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, latent_dim, kernel_size=1),
            nn.ReLU(),
            nn.ConvTranspose2d(latent_dim, 16, kernel_size=2, stride=1, padding=1),
            ResidualStack(
                in_channels=16,
                num_hiddens=16,
                num_residual_layers=num_residual_layers,
                num_residual_hiddens=num_residual_hiddens,
            ),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 8, kernel_size=2, stride=1, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(8, channels, kernel_size=2, stride=1, padding=1),
            nn.Tanh()  # Output values in range [-1, 1]
        )

    def forward(self, x):
        z_e = self.encoder(x)
        
        z_q, vq_loss, encodings = self.vq_layer(z_e)
        
        # row level mean
        counts = torch.bincount(
            encodings,
            minlength=self.num_embeddings
        ).float()
        
        avg_probs = counts / counts.sum()
        perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-10)))
        
        x_recon = self.decoder(z_q)
        return x_recon, vq_loss, perplexity

In [ ]:
def vq_vae_hyperparameter_tuning(trial) -> None:
    learning_rate = trial.suggest_float('lr', 1e-3, 1e-1)
    beta = trial.suggest_float('beta', 0.1, 0.25)
    epoch_values = 30   
    
    model = VQVAE(
        channels=1,
        latent_dim=32,
        num_embeddings=128,
        num_residual_hiddens=3,
        num_residual_layers = 16,
        beta=beta
    )
    model = model.to(device)
    
    optimizer = optim.Adam(
        model.parameters(), 
        lr=learning_rate, 
        amsgrad=True
    )
    avg_loss_total = 0
    
    for epoch in range(epoch_values):
        model.train()
        train_loss = 0
        for _, patches_adcs in enumerate(train_loader):
            patches_adcs = patches_adcs.float().to(device)

            optimizer.zero_grad()
            recon_batch, vq_loss, _ = model(patches_adcs)
        
            loss = _get_vq_loss(
                recon_batch, 
                patches_adcs, 
                vq_loss
            )
            loss.backward()
            train_loss += loss.item()
            optimizer.step()        

        avg_loss_total += train_loss / BATCH_SIZE
    
    return avg_loss_total

In [ ]:
# vq_vae_study = optuna.create_study(direction="minimize")
vq_vae_study = optuna.create_study(direction="maximize")
vq_vae_study.optimize(
    vq_vae_hyperparameter_tuning,
    TRIALS
)
print("Best hyperparameters for VQ-VAE")

for key, value in vq_vae_study.best_params.items():
    print(f"{key} = {value}")

[I 2026-03-26 14:56:38,727] A new study created in memory with name: no-name-6bf54a7e-cc68-4dd7-8ba5-aaa5642a04ea
[I 2026-03-26 14:56:54,400] Trial 0 finished with value: 292.8893766403198 and parameters: {'lr': 0.036507786452885035, 'beta': 0.11371316566500939}. Best is trial 0 with value: 292.8893766403198.
[I 2026-03-26 14:57:10,050] Trial 1 finished with value: 311.4817409515381 and parameters: {'lr': 0.06818810324720753, 'beta': 0.1520761126909115}. Best is trial 1 with value: 311.4817409515381.
[I 2026-03-26 14:57:25,572] Trial 2 finished with value: 34111863065.77823 and parameters: {'lr': 0.09546900119137143, 'beta': 0.22290485805621607}. Best is trial 2 with value: 34111863065.77823.
[I 2026-03-26 14:57:40,786] Trial 3 finished with value: 294.78356075286865 and parameters: {'lr': 0.013276850720929094, 'beta': 0.14536232064148705}. Best is trial 2 with value: 34111863065.77823.
[I 2026-03-26 14:57:56,284] Trial 4 finished with value: 293.1997413635254 and parameters: {'lr': 0.

Best hyperparameters for VQ-VAE
lr = 0.09546900119137143
beta = 0.22290485805621607
